In [121]:
import pandas as pd
from pathlib import Path
from loader import Loader
from borough_identifier import BoroughIdentifier

In [108]:
DONNEES_URL = "https://www.donneesquebec.ca/recherche/api/3/action/"
SQL_URL = "datastore_search_sql?"
URL_CRIME = "c6f482bf-bf0f-4960-8b2f-9982c211addd"

output_path = Path.cwd().resolve().parent / "data" / "raw_crime.csv"
loader = Loader()

### Data checks

In [110]:
crime_sql = f"""sql=SELECT "_id", "CATEGORIE", "DATE", "X", "Y", "LONGITUDE", "LATITUDE" from "{URL_CRIME}" where "DATE" >= '2021-01-01' AND "DATE" < '2022-01-01'"""
path = f"""{DONNEES_URL}{SQL_URL}{crime_sql}"""
path = path.replace(" ", "%20")
loader.load_to_csv(path, output_path)

In [111]:
df = pd.read_csv(output_path)

In [112]:
print(df["X"].isna().sum())
print(df["Y"].isna().sum())
print(df["LONGITUDE"].isna().sum())
print(df["LATITUDE"].isna().sum())


# Are they all NA at the same time? Yes. So we can only select longitude and latitude
all_na_filter = (df["X"].isna() & df["Y"].isna() & df["LONGITUDE"].isna() & df["LATITUDE"].isna())
print(len(df[all_na_filter]))

4278
4278
4278
4278
4278


In [113]:
print(df["DATE"].min())
print(df["DATE"].max())

2021-01-01
2021-12-31


In [114]:
df["CATEGORIE"].value_counts()

CATEGORIE
Vol dans / sur véhicule à moteur    7962
Vol de véhicule à moteur            6440
Méfait                              5820
Introduction                        5111
Vols qualifiés                      1456
Infractions entrainant la mort        34
Name: count, dtype: int64

### Preprocessing

In [122]:
crime_sql = f"""sql=SELECT "_id", "CATEGORIE", "X", "Y" from "{URL_CRIME}" where "DATE" >= '2021-01-01' AND "DATE" < '2022-01-01'"""
path = f"""{DONNEES_URL}{SQL_URL}{crime_sql}"""
path = path.replace(" ", "%20")
loader.load_to_csv(path, output_path) # Overwrite the CSV from the checks

In [123]:
df = pd.read_csv(output_path)

In [124]:
print(df["X"].isna().sum())
print(df["Y"].isna().sum())

4278
4278


In [125]:
print(f"Original dataset length: {len(df)}")
filtered_df = df.dropna(axis=0)
print(f"Filtered dataset length: {len(filtered_df)}")

dropped = len(df)-len(filtered_df)
print(f"Number of rows dropped: {dropped}")
print(f"Percentage dropped: {(dropped / len(df)) * 100:.2f}%")

Original dataset length: 26823
Filtered dataset length: 22545
Number of rows dropped: 4278
Percentage dropped: 15.95%


In [126]:
borough_identifier = BoroughIdentifier()
filtered_df["BOROUGH"] = filtered_df.apply(
    lambda row: borough_identifier.match_NAD83_to_borough(row["X"], row["Y"]), # Because custom function does not accept full Series
    axis = 1
) 
filtered_df = filtered_df.drop(columns=["X", "Y"])
filtered_df.head(10)

,_id,CATEGORIE,BOROUGH
4,161392,Introduction,Rivière-des-Prairies-Pointe-aux-Trembles
5,161406,Vol de véhicule à moteur,Rivière-des-Prairies-Pointe-aux-Trembles
6,161409,Vol dans / sur véhicule à moteur,Lachine
7,163172,Introduction,Rosemont-La Petite-Patrie
8,163179,Vols qualifiés,Rosemont-La Petite-Patrie
9,163194,Vol dans / sur véhicule à moteur,Rosemont-La Petite-Patrie
10,163195,Vol de véhicule à moteur,Rosemont-La Petite-Patrie
11,163198,Vol dans / sur véhicule à moteur,Rosemont-La Petite-Patrie
12,163203,Vol de véhicule à moteur,Rosemont-La Petite-Patrie
13,163294,Vol de véhicule à moteur,LaSalle


In [127]:
counts = filtered_df["BOROUGH"].value_counts().reset_index()
counts = counts.rename(columns={
    "count" : "criminal_acts"
})
counts

,BOROUGH,criminal_acts
0,Ville-Marie,2957
1,Mercier-Hochelaga-Maisonneuve,2081
2,Villeray-Saint-Michel-Parc-Extension,1595
3,Rosemont-La Petite-Patrie,1488
4,Côte-des-Neiges-Notre-Dame-de-Grâce,1485
5,Ahuntsic-Cartierville,1396
6,Le Plateau-Mont-Royal,1359
7,Rivière-des-Prairies-Pointe-aux-Trembles,1116
8,Montréal-Nord,1106
9,Saint-Laurent,1070


In [128]:
counts.to_csv("../data/crime_dataset.csv", index = False)